In [0]:
%run /Users/dubeyyash10@outlook.com/config

In [0]:
import boto3

s3 = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_DEFAULT_REGION
)

response = s3.list_objects_v2(Bucket='medinsight-partd-yash')
for obj in response.get('Contents', []):
    print(obj['Key'])

checkpoints/
delta/
raw/
raw/year=2021/partd_2021.csv


In [0]:
import boto3
import pandas as pd
import io

s3 = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_DEFAULT_REGION
)

# Read first 100 rows only to check structure
obj = s3.get_object(Bucket='medinsight-partd-yash', Key='raw/year=2021/partd_2021.csv')
df = pd.read_csv(obj['Body'], nrows=100)

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nNull counts:\n", df.isnull().sum())
print("\nSample:\n", df.head(3))

Shape: (100, 22)

Columns: ['Prscrbr_NPI', 'Prscrbr_Last_Org_Name', 'Prscrbr_First_Name', 'Prscrbr_City', 'Prscrbr_State_Abrvtn', 'Prscrbr_State_FIPS', 'Prscrbr_Type', 'Prscrbr_Type_Src', 'Brnd_Name', 'Gnrc_Name', 'Tot_Clms', 'Tot_30day_Fills', 'Tot_Day_Suply', 'Tot_Drug_Cst', 'Tot_Benes', 'GE65_Sprsn_Flag', 'GE65_Tot_Clms', 'GE65_Tot_30day_Fills', 'GE65_Tot_Drug_Cst', 'GE65_Tot_Day_Suply', 'GE65_Bene_Sprsn_Flag', 'GE65_Tot_Benes']

Null counts:
 Prscrbr_NPI               0
Prscrbr_Last_Org_Name     0
Prscrbr_First_Name        0
Prscrbr_City              0
Prscrbr_State_Abrvtn      0
Prscrbr_State_FIPS        0
Prscrbr_Type              0
Prscrbr_Type_Src          0
Brnd_Name                 0
Gnrc_Name                 0
Tot_Clms                  0
Tot_30day_Fills           0
Tot_Day_Suply             0
Tot_Drug_Cst              0
Tot_Benes                50
GE65_Sprsn_Flag          62
GE65_Tot_Clms            38
GE65_Tot_30day_Fills     38
GE65_Tot_Drug_Cst        38
GE65_Tot_Day_Supl

In [0]:
obj = s3.get_object(Bucket='medinsight-partd-yash', Key='raw/year=2021/partd_2021.csv')
df_full = pd.read_csv(obj['Body'])
print("Total rows:", f"{len(df_full):,}")
print("Total columns:", len(df_full.columns))

/home/spark-a26e04be-00a4-4573-84ee-2f/.ipykernel/2117/command-6802264116948038-3644773802:2: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_full = pd.read_csv(obj['Body'])


Total rows: 25,231,862
Total columns: 22


In [0]:
print(df_full.isnull().sum())

Prscrbr_NPI                     0
Prscrbr_Last_Org_Name           0
Prscrbr_First_Name             11
Prscrbr_City                    2
Prscrbr_State_Abrvtn            0
Prscrbr_State_FIPS              0
Prscrbr_Type                   56
Prscrbr_Type_Src                0
Brnd_Name                       0
Gnrc_Name                       0
Tot_Clms                        0
Tot_30day_Fills                 0
Tot_Day_Suply                   0
Tot_Drug_Cst                    0
Tot_Benes                14503885
GE65_Sprsn_Flag          13996182
GE65_Tot_Clms            11235680
GE65_Tot_30day_Fills     11235680
GE65_Tot_Drug_Cst        11235680
GE65_Tot_Day_Suply       11235680
GE65_Bene_Sprsn_Flag      2655545
GE65_Tot_Benes           22576317
dtype: int64


In [0]:
# Columns we will use in the pipeline
cols_to_keep = [
    'Prscrbr_NPI', 'Prscrbr_Last_Org_Name', 'Prscrbr_First_Name',
    'Prscrbr_State_Abrvtn', 'Prscrbr_Type', 'Brnd_Name',
    'Gnrc_Name', 'Tot_Clms', 'Tot_Day_Suply', 'Tot_Drug_Cst'
]

print("Columns we keep:", len(cols_to_keep))
print("Columns we drop:", 22 - len(cols_to_keep))
print("Dropped:", [c for c in df_full.columns if c not in cols_to_keep])

Columns we keep: 10
Columns we drop: 12
Dropped: ['Prscrbr_City', 'Prscrbr_State_FIPS', 'Prscrbr_Type_Src', 'Tot_30day_Fills', 'Tot_Benes', 'GE65_Sprsn_Flag', 'GE65_Tot_Clms', 'GE65_Tot_30day_Fills', 'GE65_Tot_Drug_Cst', 'GE65_Tot_Day_Suply', 'GE65_Bene_Sprsn_Flag', 'GE65_Tot_Benes']
